# Plot hourly FE comparison under calibrated mutation

Reads `Hourly_FE_Index.csv` and saved hourly FE files from:

```text
Code_Submission/simulation_mutation/Output files (Risk Neutral, Mutation, Verified)
```

Use the first control section to select match, mutation family, party, and optional calibrated final physical-shift levels.

In [ ]:
# Plot hourly FE comparison under calibrated correlation mutation
# ============================================================
# Run this after mutation_simulation_from_samples.ipynb.
# It plots FE distributions for one match as baseline + calibrated mutation levels.
# Hourly FE files generated by the compact simulation notebook are rounded to two decimals by default.

from __future__ import annotations

from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde


# ============================================================
# Required inputs
# ============================================================
output_dir = str(Path("simulation_mutation") / "Output files (Risk Neutral, Mutation, Verified)")
match_id = 1

# Which mutation family to compare:
#   "shape", "basis", "load_price", "cannibalization"
mutation_family_to_plot = "shape"

# Optional calibrated final physical-shift filter. Leave None to include all available levels.
# Examples: [-0.10, -0.30, -0.50] for shape/basis/cannibalization; [0.10, 0.30, 0.50] for load_price.
filter_target_physical_shifts = None

# Options: "both", "seller", "buyer"
plot_party = "both"

# Risk parameters used only for plotting the risk-adjusted FE line.
lambda_seller = 0.0
lambda_buyer = 0.0

# Plot controls
clip_for_plot = True
lower_q = 0.005
upper_q = 0.995
common_x_axis = True

plot_histogram = True
plot_density = True
hist_bins = 80

figsize = (14, 10)
tick_font_size = 12
axis_title_font_size = 14
subplot_title_font_size = 14
legend_font_size = 11

save_fig = True
save_fig_dir_name = "Figures_Hourly_FE_Mutation"

SETTINGS_FE = {
    "output_dir": output_dir,
    "match_id": int(match_id),
    "mutation_family_to_plot": mutation_family_to_plot,
    "filter_target_physical_shifts": filter_target_physical_shifts,
    "plot_party": plot_party,
    "lambda_seller": float(lambda_seller),
    "lambda_buyer": float(lambda_buyer),
    "clip_for_plot": bool(clip_for_plot),
    "lower_q": float(lower_q),
    "upper_q": float(upper_q),
    "common_x_axis": bool(common_x_axis),
    "plot_histogram": bool(plot_histogram),
    "plot_density": bool(plot_density),
    "hist_bins": int(hist_bins),
    "figsize": figsize,
    "tick_font_size": tick_font_size,
    "axis_title_font_size": axis_title_font_size,
    "subplot_title_font_size": subplot_title_font_size,
    "legend_font_size": legend_font_size,
    "save_fig": bool(save_fig),
    "save_fig_dir_name": save_fig_dir_name,
}


NOTEBOOK_CWD = Path.cwd().resolve()
BUNDLE_ROOT = NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == "simulation_mutation" else NOTEBOOK_CWD


def _search_roots(max_parent_depth: int = 4) -> list[Path]:
    roots = []
    seen = set()
    for anchor in [BUNDLE_ROOT, NOTEBOOK_CWD, Path("/mnt/data")]:
        p = Path(anchor).expanduser()
        for root in [p, *list(p.parents)[:max_parent_depth]]:
            key = str(root)
            if key not in seen:
                seen.add(key)
                roots.append(root)
    return roots


def _integer_stat_label(value: float) -> str:
    if pd.isna(value) or not np.isfinite(value):
        return "NA"
    return f"{int(np.rint(float(value))):,}"


def _output_dir_path(output_dir: str | Path) -> Path:
    path = Path(output_dir).expanduser()
    if path.exists():
        return path.resolve()
    for root in _search_roots():
        alt = root / path
        if alt.exists():
            return alt.resolve()
    raise FileNotFoundError(f"Output directory was not found: {output_dir}")


def _hourly_fe_index_file(output_dir: str | Path) -> Path:
    out_dir = _output_dir_path(output_dir)
    index_file = out_dir / "Hourly_FE_Index.csv"
    if not index_file.exists():
        raise FileNotFoundError(f"Hourly FE index file not found: {index_file}")
    return index_file


def list_saved_hourly_fe_scenarios(match_id: int, output_dir: str | Path) -> pd.DataFrame:
    index_df = read_csv_optimized(_hourly_fe_index_file(output_dir))
    index_df["match_id"] = pd.to_numeric(index_df["match_id"], errors="coerce").astype("Int64")
    out = index_df.loc[index_df["match_id"] == int(match_id)].copy()
    return out.sort_values("scenario_order").reset_index(drop=True)


def _as_float_filter(values):
    if values is None:
        return None
    if isinstance(values, (int, float, np.integer, np.floating)):
        return [float(values)]
    return [float(x) for x in values]


def _numeric_filter_mask(series: pd.Series, allowed_values, atol: float = 1e-9) -> pd.Series:
    allowed = _as_float_filter(allowed_values)
    if allowed is None:
        return pd.Series(True, index=series.index)
    numeric = pd.to_numeric(series, errors="coerce")
    keep = pd.Series(False, index=series.index)
    for value in allowed:
        keep = keep | np.isclose(numeric, float(value), atol=atol)
    return keep


def select_family_scenarios(index_df: pd.DataFrame,
                            family: str,
                            target_physical_shifts=None) -> pd.DataFrame:
    baseline_mask = index_df["scenario_type"].astype(str).eq("baseline")
    family_mask = index_df["mutation_family"].astype(str).eq(str(family))
    df = index_df.loc[baseline_mask | family_mask].copy()

    if target_physical_shifts is not None and "target_physical_shift" in df.columns:
        baseline_mask = df["scenario_type"].astype(str).eq("baseline")
        df = df.loc[baseline_mask | _numeric_filter_mask(df["target_physical_shift"], target_physical_shifts)].copy()

    if df.empty:
        raise ValueError(f"No baseline/family scenarios found for mutation family {family!r}.")
    return df.sort_values("scenario_order").reset_index(drop=True)


def load_saved_hourly_fe_from_index_row(index_row: pd.Series, output_dir: str | Path) -> pd.DataFrame:
    fe_file = Path(index_row["hourly_fe_file"]).expanduser()
    if not fe_file.exists():
        out_dir = _output_dir_path(output_dir)
        fe_file = out_dir / "Hourly_FE" / fe_file.name
    if not fe_file.exists():
        raise FileNotFoundError(f"Saved hourly FE file not found: {fe_file}")

    hourly_fe_df = read_csv_optimized(fe_file)
    hourly_fe_df["replication"] = pd.to_numeric(hourly_fe_df["replication"], errors="coerce").astype("Int64")
    hourly_fe_df["hour_index"] = pd.to_numeric(hourly_fe_df["hour_index"], errors="coerce").astype("Int64")
    return hourly_fe_df


def _build_ppa_label(index_row: pd.Series) -> str:
    ppa_type = str(index_row.get("ppa_type", "Unknown"))
    profile_type = str(index_row.get("profile_type", "Unknown"))
    strike_price = pd.to_numeric(pd.Series([index_row.get("strike_price_mwh", np.nan)]), errors="coerce").iloc[0]
    fixed_volume = pd.to_numeric(pd.Series([index_row.get("volume_mw", np.nan)]), errors="coerce").iloc[0]

    parts = [f"{ppa_type}-{profile_type}"]
    if pd.notna(strike_price):
        parts.append(f"Strike = {_integer_stat_label(strike_price)}")
    if str(profile_type).strip().lower() == "fix" and pd.notna(fixed_volume):
        parts.append(f"Volume = {_integer_stat_label(fixed_volume)}")
    return " | ".join(parts)


def scenario_panel_title(index_row: pd.Series) -> str:
    if str(index_row.get("scenario_type", "")) == "baseline":
        return "Baseline"

    physical_shift = pd.to_numeric(pd.Series([index_row.get("target_physical_shift", np.nan)]), errors="coerce").iloc[0]
    latent_delta = pd.to_numeric(pd.Series([index_row.get("target_delta", np.nan)]), errors="coerce").iloc[0]

    if pd.notna(physical_shift):
        return f"Target physical $\\Delta\\rho$ = {physical_shift:+.2f}"
    if pd.notna(latent_delta):
        return f"Latent $\\Delta\\rho$ = {latent_delta:+.2f}"
    return str(index_row.get("mutation_level_label", index_row.get("scenario_name", "")))


def _build_plot_series(series: pd.Series,
                       clip_for_plot: bool = True,
                       lower_q: float = 0.005,
                       upper_q: float = 0.995) -> Tuple[pd.Series, float, float]:
    series = pd.to_numeric(pd.Series(series), errors="coerce").dropna()
    if series.empty:
        return series, np.nan, np.nan

    if (not clip_for_plot) or (series.nunique() <= 1):
        return series, float(series.min()), float(series.max())

    x_low = float(series.quantile(lower_q))
    x_high = float(series.quantile(upper_q))
    if not np.isfinite(x_low) or not np.isfinite(x_high) or x_high <= x_low:
        return series, float(series.min()), float(series.max())

    plot_series = series[(series >= x_low) & (series <= x_high)].copy()
    if plot_series.empty:
        return series, float(series.min()), float(series.max())

    return plot_series, x_low, x_high


def _panel_data_for_party(index_rows: pd.DataFrame,
                          output_dir: str | Path,
                          party_label: str,
                          lambda_value: float,
                          settings: dict):
    column_name = "seller_fe" if str(party_label).strip().lower() == "seller" else "buyer_fe"
    panels = []

    for _, row in index_rows.iterrows():
        hourly_fe_df = load_saved_hourly_fe_from_index_row(row, output_dir)
        series = pd.to_numeric(hourly_fe_df[column_name], errors="coerce").dropna()
        plot_series, x_low, x_high = _build_plot_series(
            series,
            clip_for_plot=settings["clip_for_plot"],
            lower_q=settings["lower_q"],
            upper_q=settings["upper_q"],
        )
        mean_val = float(series.mean()) if not series.empty else np.nan
        std_val = float(series.std(ddof=0)) if not series.empty else np.nan
        risk_line = mean_val + float(lambda_value) * std_val if pd.notna(mean_val) and pd.notna(std_val) else np.nan

        panels.append({
            "index_row": row,
            "series": series,
            "plot_series": plot_series,
            "x_low": x_low,
            "x_high": x_high,
            "mean_val": mean_val,
            "std_val": std_val,
            "risk_line": risk_line,
        })

    return panels


def plot_fe_comparison(index_rows: pd.DataFrame,
                       output_dir: str | Path,
                       match_id: int,
                       mutation_family: str,
                       party_label: str,
                       lambda_value: float,
                       settings: dict) -> None:
    panels = _panel_data_for_party(index_rows, output_dir, party_label, lambda_value, settings)
    n_panels = len(panels)
    n_cols = 2 if n_panels > 1 else 1
    n_rows = int(np.ceil(n_panels / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=settings["figsize"], squeeze=False)
    axes_flat = axes.ravel()

    common_low, common_high = np.inf, -np.inf
    if settings["common_x_axis"]:
        for panel in panels:
            for value in [panel["x_low"], panel["x_high"], panel["mean_val"], panel["risk_line"], 0.0]:
                if pd.notna(value) and np.isfinite(value):
                    common_low = min(common_low, float(value))
                    common_high = max(common_high, float(value))
        if np.isfinite(common_low) and np.isfinite(common_high) and common_high > common_low:
            pad = 0.05 * (common_high - common_low)
            common_low -= pad
            common_high += pad
        else:
            common_low, common_high = None, None
    else:
        common_low, common_high = None, None

    for panel_idx, panel in enumerate(panels):
        ax = axes_flat[panel_idx]
        index_row = panel["index_row"]
        plot_series = panel["plot_series"]
        mean_val = panel["mean_val"]
        risk_line = panel["risk_line"]

        if plot_series.empty:
            ax.set_title(f"{scenario_panel_title(index_row)}\nNo valid FE values", fontsize=settings["subplot_title_font_size"])
            continue

        x_low = common_low if common_low is not None else panel["x_low"]
        x_high = common_high if common_high is not None else panel["x_high"]
        x_low = min(x_low, mean_val, risk_line, 0.0)
        x_high = max(x_high, mean_val, risk_line, 0.0)
        if x_high <= x_low:
            x_low -= 1.0
            x_high += 1.0

        if settings["plot_histogram"]:
            ax.hist(plot_series, bins=int(settings["hist_bins"]), density=True, alpha=0.35, label="Histogram")

        if settings["plot_density"] and plot_series.nunique() > 1:
            kde = gaussian_kde(plot_series)
            x_grid = np.linspace(x_low, x_high, 600)
            y_grid = kde(x_grid)
            ax.plot(x_grid, y_grid, linewidth=2, label="Density curve")

        ax.axvline(mean_val, linestyle="--", linewidth=2, label=f"Mean = {_integer_stat_label(mean_val)}")
        ax.axvline(risk_line, linestyle="-.", linewidth=2, label=f"Risk-adjusted FE = {_integer_stat_label(risk_line)}")
        ax.axvline(0.0, linestyle="-", linewidth=1.5, label="_nolegend_")

        ax.set_xlim(x_low, x_high)
        ax.set_title(
            f"{scenario_panel_title(index_row)}\n{_build_ppa_label(index_row)}",
            fontsize=settings["subplot_title_font_size"],
        )
        ax.set_xlabel(f"{party_label} hourly FE", fontsize=settings["axis_title_font_size"])
        ax.set_ylabel("Probability density", fontsize=settings["axis_title_font_size"])
        ax.tick_params(axis="both", labelsize=settings["tick_font_size"])
        ax.legend(fontsize=settings["legend_font_size"])

        print(
            f"{party_label} | {scenario_panel_title(index_row)} | "
            f"Mean={_integer_stat_label(mean_val)} | "
            f"Std={_integer_stat_label(panel['std_val'])} | "
            f"Risk-adjusted={_integer_stat_label(risk_line)} | "
            f"{_build_ppa_label(index_row)}"
        )

    for ax in axes_flat[n_panels:]:
        ax.axis("off")

    fig.tight_layout()

    if settings["save_fig"]:
        out_dir = _output_dir_path(output_dir) / settings["save_fig_dir_name"]
        out_dir.mkdir(parents=True, exist_ok=True)
        fig_path = out_dir / f"Hourly_FE_Comparison_Match_{int(match_id):04d}__{mutation_family}__{party_label}.png"
        fig.savefig(fig_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure: {fig_path}")

    plt.show()


# ============================================================
# Execute
# ============================================================
fe_index_df = list_saved_hourly_fe_scenarios(SETTINGS_FE["match_id"], SETTINGS_FE["output_dir"])
family_index_df = select_family_scenarios(
    fe_index_df,
    SETTINGS_FE["mutation_family_to_plot"],
    target_physical_shifts=SETTINGS_FE["filter_target_physical_shifts"],
)

display_cols = [
    "scenario_name", "scenario_type", "mutation_family", "target_physical_shift",
    "calibrated_latent_delta", "target_delta", "ppa_type", "profile_type",
    "strike_price_mwh", "volume_mw",
]
display_cols = [c for c in display_cols if c in fe_index_df.columns]

print("Available scenarios for this match:")
print(fe_index_df[display_cols].to_string(index=False))

print("\nSelected scenarios for FE comparison:")
print(family_index_df[[c for c in display_cols if c in family_index_df.columns]].to_string(index=False))

plot_mode = str(SETTINGS_FE["plot_party"]).strip().lower()
if plot_mode not in {"both", "seller", "buyer"}:
    raise ValueError("plot_party must be 'both', 'seller', or 'buyer'.")

if plot_mode in {"both", "seller"}:
    plot_fe_comparison(
        index_rows=family_index_df,
        output_dir=SETTINGS_FE["output_dir"],
        match_id=SETTINGS_FE["match_id"],
        mutation_family=SETTINGS_FE["mutation_family_to_plot"],
        party_label="Seller",
        lambda_value=SETTINGS_FE["lambda_seller"],
        settings=SETTINGS_FE,
    )

if plot_mode in {"both", "buyer"}:
    plot_fe_comparison(
        index_rows=family_index_df,
        output_dir=SETTINGS_FE["output_dir"],
        match_id=SETTINGS_FE["match_id"],
        mutation_family=SETTINGS_FE["mutation_family_to_plot"],
        party_label="Buyer",
        lambda_value=SETTINGS_FE["lambda_buyer"],
        settings=SETTINGS_FE,
    )

In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.
import numpy as np

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))
